# ICWSM 2026 Final Framing Analysis

This cleaned notebook runs the final aggregate analysis for the ICWSM 2026 COVID framing paper. It uses the July 17, 2025 derived LLTR CSV files and delegates the reusable logic to `src/final_framing_analysis.py`.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT / "src"))

from final_framing_analysis import (
    DEFAULT_DOC_PROBS,
    DEFAULT_FULL_DIST,
    compute_lexical_jsd,
    compute_relational_jsd,
    compute_source_topic_distribution,
    save_jsd_heatmap,
)


## Load Derived Data

The public repository uses the latest July 17, 2025 derived result group. Topic 6 is discussed in the README as an example, but the analysis below covers all eight topics.


In [ ]:
import pandas as pd

full_dist = pd.read_csv(REPO_ROOT / DEFAULT_FULL_DIST)
doc_probs = pd.read_csv(REPO_ROOT / DEFAULT_DOC_PROBS)

full_dist.shape, doc_probs.shape


## Lexical Jensen-Shannon Divergence

This baseline compares topic-word distributions after reweighting by source usage.


In [ ]:
lexical_jsd = compute_lexical_jsd(full_dist)
lexical_jsd


## Syntactic/Relational Jensen-Shannon Divergence

This is the main LLTR-based analysis. It compares source-specific distributions over relation-argument evidence for each topic.


In [ ]:
relational_jsd_top50 = compute_relational_jsd(full_dist, top_k=50)
relational_jsd_top100 = compute_relational_jsd(full_dist, top_k=100)
relational_jsd_all = compute_relational_jsd(full_dist, top_k=None)

relational_jsd_top100


## Source-Topic Distribution

This summarizes each source's contribution to each topic using document-topic probabilities.


In [ ]:
source_topic_distribution = compute_source_topic_distribution(doc_probs)
source_topic_distribution


## Save Results and Figures

The public script performs this step automatically. The notebook keeps the same output paths for easy inspection.


In [ ]:
RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

lexical_jsd.to_csv(RESULTS_DIR / "lexical_jsd.csv", index=False)
relational_jsd_top50.to_csv(RESULTS_DIR / "relational_jsd_top50.csv", index=False)
relational_jsd_top100.to_csv(RESULTS_DIR / "relational_jsd_top100.csv", index=False)
relational_jsd_all.to_csv(RESULTS_DIR / "relational_jsd_all.csv", index=False)
source_topic_distribution.to_csv(RESULTS_DIR / "source_topic_distribution.csv", index=False)

save_jsd_heatmap(
    lexical_jsd,
    RESULTS_DIR / "lexical_jsd_heatmap.png",
    title="Lexical Topic-Word JSD Across Sources",
)
save_jsd_heatmap(
    relational_jsd_top100,
    RESULTS_DIR / "relational_jsd_top100_heatmap.png",
    title="LLTR Relational JSD Across Sources (top100)",
)
